
# THEMIS dust IR: radiation-field slope (alpha)

The Jones et al. (2017) THEMIS dust model distributes grains over a range of
starlight intensities $U$ with a power law $dU/dM \propto U^{-\alpha}$.
The slope ``alpha`` controls how much warm, intensely-illuminated dust
contributes relative to the cold diffuse component: a *smaller* alpha puts more
mass at high $U$, shifting the FIR peak blueward and filling in the
mid-IR.

tengri ships the FSPS/DustEM THEMIS templates (alpha=2.0); the alpha axis is
added by re-shaping them with CIGALE's DustEM alpha grid, anchored so that
``alpha = 2.0`` reproduces the FSPS template exactly
(``scripts/build_themis_alpha_axis.py``). This sweeps alpha at fixed grain
composition and radiation-field minimum.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import warnings

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.plot import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

C_AA_PER_S = 2.998e18
ssp = tengri.load_ssp()

model = tengri.SEDModel.build(
    ssp,
    sfh={"type": "const", "all_params": tengri.Fixed(tengri.DEFAULT), "log_total_mass": 11.0},
    dust_attenuation={
        "law": "power_law",
        "type": "two_component",
        "all_params": tengri.Fixed(tengri.DEFAULT),
        "tau_diff": 1.0,
        "tau_bc": 0.3,
    },
    dust_emission={
        "type": "themis",
        "all_params": tengri.Fixed(tengri.DEFAULT),
        "dust_gamma_dl": 0.1,
    },
    redshift=tengri.Fixed(0.05),
)
p0 = dict(model.spec.sample(jax.random.PRNGKey(0)))

alpha_values = [1.0, 1.5, 2.0, 2.5, 3.0]
colors = plt.cm.inferno(np.linspace(0.15, 0.8, len(alpha_values)))

fig, ax = plt.subplots(figsize=(7.2, 4.6))

for a, c in zip(alpha_values, colors):
    out = model.predict({**p0, "dust_alpha": jnp.float64(a)})
    wave = np.asarray(model.wavelengths)
    nu_l_nu = C_AA_PER_S / wave * np.asarray(out.rest_sed())
    lw = 2.6 if a == 2.0 else 1.8
    ax.loglog(
        wave, nu_l_nu, color=c, lw=lw, label=rf"$\alpha={a:.1f}$" + (" (FSPS)" if a == 2.0 else "")
    )

ax.set(
    xlim=(3e4, 1e7),
    ylim=(1e42, 1e45),
    xlabel=r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]",
    ylabel=r"$\nu L_\nu$  [erg s$^{-1}$]",
)
ax.legend(frameon=False, fontsize=9, loc="upper right", title=r"$dU/dM\propto U^{-\alpha}$")

fig.tight_layout()
plt.savefig("plot_themis_alpha_sweep.png", dpi=150, bbox_inches="tight")